## 1) Train model

In [ ]:
%load_ext autoreload
%autoreload 2

# Internal import
import deep_learning_land_use_classification.config as dataset
import deep_learning_land_use_classification.evaluation as evaluation

# External imports
import torch
import torch.nn as nn
from torchvision import models, transforms
from torch.utils.data import DataLoader, Dataset
from PIL import Image
import numpy as np

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


We use BCEWithLogitsLoss because our goal is multiclass classification. We also penalize larger classes using pos_weight due to the class imbalance present in the dataset. 

In [7]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # handles inconsistent sizes
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std =[0.229, 0.224, 0.225]
    )
])

In [8]:
class MultiLabelDataset(Dataset):
    def __init__(self, dataframe, class_names, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.class_names = class_names
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image = Image.open(row["path"]).convert("RGB")

        if self.transform:
            image = self.transform(image)

        label = torch.tensor(row[self.class_names].values.astype(np.float32))

        return image, label

In [ ]:
train_dataset = MultiLabelDataset(dataset.train_df, dataset.class_names, dataset.transform)
test_dataset  = MultiLabelDataset(dataset.test_df, dataset.class_names, dataset.transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print ("Using device:", device)

model = models.resnet50(pretrained=True)

# Replace final layer
model.fc = nn.Linear(model.fc.in_features, dataset.num_classes)

model = model.to(device)

Using device: cuda


c:\Users\tomer\anaconda3\envs\Deep-learning-land-use-classification\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\tomer\anaconda3\envs\Deep-learning-land-use-classification\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [ ]:
labels = dataset.train_df[dataset.class_names].values

# Count positives per class
pos_counts = labels.sum(axis=0)
neg_counts = len(labels) - pos_counts

# Avoid division by zero
pos_weight = torch.tensor(neg_counts / (pos_counts + 1e-5), dtype=torch.float32)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(device))
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

In [12]:
def train(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

In [13]:
def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item()

    return total_loss / len(loader)

In [14]:
epochs = 5

for epoch in range(epochs):
    train_loss = train(model, train_loader, optimizer, criterion)
    test_loss  = evaluate(model, test_loader, criterion)

    print(f"Epoch {epoch+1}/{epochs}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Test Loss:  {test_loss:.4f}")

Epoch 1/5
Train Loss: 0.5303
Test Loss:  0.2414
Epoch 2/5
Train Loss: 0.2042
Test Loss:  0.1594
Epoch 3/5
Train Loss: 0.1408
Test Loss:  0.1452
Epoch 4/5
Train Loss: 0.1117
Test Loss:  0.1428
Epoch 5/5
Train Loss: 0.0872
Test Loss:  0.1577


In [28]:
precision, recall, f1, p_macro, r_macro, f1_macro = evaluation.compute_accuracy_metrics(
    model,
    test_loader,
    device,
    threshold=0.5
)

for i, cls in enumerate(dataset.class_names):
    print(f"{cls:15s} | Precision: {precision[i]:.3f} | Recall: {recall[i]:.3f} | F1: {f1[i]:.3f}")

print("\n--- Macro Averages ---")
print(f"Precision: {p_macro:.3f}")
print(f"Recall:    {r_macro:.3f}")
print(f"F1 Score:  {f1_macro:.3f}")

airplane        | Precision: 1.000 | Recall: 1.000 | F1: 1.000
bare-soil       | Precision: 0.726 | Recall: 0.850 | F1: 0.783
buildings       | Precision: 0.832 | Recall: 0.976 | F1: 0.899
cars            | Precision: 0.871 | Recall: 0.856 | F1: 0.864
chaparral       | Precision: 0.871 | Recall: 1.000 | F1: 0.931
court           | Precision: 0.917 | Recall: 0.957 | F1: 0.936
dock            | Precision: 1.000 | Recall: 1.000 | F1: 1.000
field           | Precision: 1.000 | Recall: 1.000 | F1: 1.000
grass           | Precision: 0.919 | Recall: 0.896 | F1: 0.907
mobile-home     | Precision: 0.964 | Recall: 1.000 | F1: 0.982
pavement        | Precision: 0.938 | Recall: 0.930 | F1: 0.934
sand            | Precision: 0.895 | Recall: 0.879 | F1: 0.887
sea             | Precision: 1.000 | Recall: 1.000 | F1: 1.000
ship            | Precision: 1.000 | Recall: 1.000 | F1: 1.000
tanks           | Precision: 1.000 | Recall: 0.889 | F1: 0.941
trees           | Precision: 0.906 | Recall: 0.870 | F1